# CS4.406 A1 — EB-NeRD: lexical + semantic retrieval, ranking, Codabench submission

**What this notebook produces:** `predictions.zip` for the
[RecSys 2024 Challenge / Codabench 2469](https://www.codabench.org/competitions/2469/),
plus the Q1–Q5 artefacts (feature store, BM25 recall@K, embedding recall@K, evaluation
harness with slices + bootstrap CIs, leakage tests, ablation).

It is the deliberate twin of the MIND notebook: same core library, same featuriser
contract, same metric harness. Only the I/O layer and a handful of dataset-specific
features differ. Whatever conclusion survives both datasets is a conclusion about news
recommendation; whatever does not is a conclusion about one publisher.

**How to run on Kaggle (free tier)**
1. Attach the EB-NeRD data as Kaggle datasets: `ebnerd_large` (train + validation +
   `articles.parquet`) and `ebnerd_testset`. Optionally attach the pre-trained
   `Ekstra_Bladet_word2vec` / `google_bert_base_multilingual_cased` artefacts — they are
   auto-detected, and if absent the notebook encodes the Danish headlines itself.
2. *Settings*: `Accelerator = GPU T4 x2`, `Internet = On` (only needed if you want the
   multilingual encoder downloaded).
3. Run all. Expected wall-clock: **~3.5–4.5 h**, dominated by the 13.5 M-impression test
   pass. Nothing here needs more than ~12 GB RAM.
4. Download `/kaggle/working/predictions.zip` and upload it to Codabench.

**Scale reality check.** The test set is 13,536,710 impressions; the beyond-accuracy
subset carries 250 candidates each, so the exploded table is >150 M rows. Nothing in this
notebook ever materialises it. Every pass over the data is chunked by a *pair budget*
and every article/user attribute is an array you index into.

## 0. Configuration

In [4]:
from pathlib import Path
for p in sorted(Path("/kaggle/input").rglob("*")):
    if p.is_file() and p.suffix in {".parquet", ".zip"}:
        print(f"{p.stat().st_size/1e6:8.1f} MB  {p}")

   150.8 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/articles.parquet
   541.9 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/train/behaviors.parquet
  1240.5 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/train/history.parquet
   586.5 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/validation/behaviors.parquet
  1125.3 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/validation/history.parquet
   150.8 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_testset/ebnerd_testset/articles.parquet
   567.8 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_testset/ebnerd_testset/test/behaviors.parquet
  1157.7 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_testset/ebnerd_testset/test/history.parquet
   153.6 MB  /kaggle/input/datasets/sherleysonali/ebnerd-word2vec/Ekstra_Bladet_word2vec/document_vector.parquet


In [5]:
import gc
import json
import os
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import polars as pl
import pyarrow.parquet as pq
import scipy.sparse as sp

warnings.filterwarnings("ignore", category=DeprecationWarning)
TEST_MODE = os.environ.get("A1_TEST_MODE") == "1"

CFG = dict(
    seed=42,
    train_mod=(25, 4),             # keep impression_id % 25 < 2  ≈ 8 % of the train file
    eval_mod=(250, 2),             # ≈ 0.8 % of the validation file — enough for 80 k impressions
    train_impressions=700_000,
    es_impressions=120_000,
    eval_impressions=80_000,
    recall_queries=3_000,
    hist_len=30,                   # long-term profile
    hist_recent=5,                 # short-term profile
    hist_member=50,                # horizon for the "already read it" test
    dim=64,
    tfidf_min_df=3,
    num_boost_round=600,
    learning_rate=0.06,
    num_leaves=63,
    batch_rows=400_000,            # parquet rows per streaming batch
    pair_budget=1_500_000,
    out_dir="/kaggle/working",
    input_root="/kaggle/input",
)

if TEST_MODE:
    CFG.update(
        train_mod=(1, 1), eval_mod=(1, 1), train_impressions=400, es_impressions=100, eval_impressions=200,
        recall_queries=40, num_boost_round=30, batch_rows=500, pair_budget=5_000,
        dim=8, tfidf_min_df=1,
        out_dir=os.environ.get("A1_OUT", "/tmp/out_eb"),
        input_root=os.environ.get("A1_DATA", "/tmp/ebnerd"),
    )

Path(CFG["out_dir"]).mkdir(parents=True, exist_ok=True)
np.random.seed(CFG["seed"])
N_THREADS = max(1, os.cpu_count() or 4)
print(f"polars {pl.__version__} | threads {N_THREADS} | test_mode={TEST_MODE}")


def ensure(pkg, module=None):
    """Install a soft dependency only if it is actually missing (Kaggle usually has it)."""
    try:
        __import__(module or pkg.replace("-", "_"))
        return True
    except ImportError:
        if TEST_MODE:
            return False
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
        try:
            __import__(module or pkg.replace("-", "_"))
            return True
        except ImportError:
            return False


ensure("sentence-transformers", "sentence_transformers")


def tic(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)
    return time.time()


def toc(t0, msg=""):
    print(f"    ...{msg} {time.time() - t0:.1f}s", flush=True)

polars 1.35.2 | threads 4 | test_mode=False


## Core library (identical to the MIND notebook)

In [6]:
import numpy as np
import scipy.sparse as sp

# --------------------------------------------------------------------------
# 1. Article-side text representations
# --------------------------------------------------------------------------


def build_tfidf(texts, stop_words=None, min_df=3, max_features=300_000):
    """TF-IDF matrix over the article catalogue (docs x vocab), L2-normalised."""
    from sklearn.feature_extraction.text import TfidfVectorizer

    vec = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words=stop_words,
        min_df=min_df,
        max_features=max_features,
        sublinear_tf=True,
        dtype=np.float32,
    )
    X = vec.fit_transform(texts)
    return vec, X.tocsr()


def build_bm25(texts, stop_words=None, min_df=3, max_features=300_000, k1=1.2, b=0.75):
    """
    Classic Okapi BM25 document-side weights as a sparse (docs x vocab) CSR.

    w(d,t) = idf(t) * tf(d,t)*(k1+1) / (tf(d,t) + k1*(1 - b + b*len(d)/avgdl))

    A query is then scored with a single sparse mat-mul: scores = Q @ W.T,
    where Q holds the query-side term frequencies. This is the inverted index —
    CSC/CSR storage *is* a postings list, and scipy does the merge in C.
    """
    from sklearn.feature_extraction.text import CountVectorizer

    vec = CountVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words=stop_words,
        min_df=min_df,
        max_features=max_features,
        dtype=np.int32,
    )
    C = vec.fit_transform(texts).tocsr()
    n_docs = C.shape[0]

    df = np.bincount(C.indices, minlength=C.shape[1]).astype(np.float32)
    idf = np.log(1.0 + (n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)

    dl = np.asarray(C.sum(axis=1)).ravel().astype(np.float32)
    avgdl = float(dl.mean()) if dl.mean() > 0 else 1.0

    W = C.astype(np.float32).tocsr()
    tf = W.data
    # per-nonzero document length
    rows = np.repeat(np.arange(n_docs, dtype=np.int64), np.diff(W.indptr))
    denom = tf + k1 * (1.0 - b + b * dl[rows] / avgdl)
    W.data = idf[W.indices] * tf * (k1 + 1.0) / denom
    return vec, W, idf


def svd_reduce(X, n_components=64, seed=0):
    """Truncated SVD (LSA) + L2 normalisation. Dense float32 (n x d)."""
    from sklearn.decomposition import TruncatedSVD

    n_components = int(min(n_components, max(2, min(X.shape) - 1)))
    svd = TruncatedSVD(n_components=n_components, random_state=seed, algorithm="randomized")
    Z = svd.fit_transform(X).astype(np.float32)
    return l2_normalise(Z)


def l2_normalise(Z):
    Z = np.asarray(Z, dtype=np.float32)
    n = np.linalg.norm(Z, axis=1, keepdims=True)
    np.maximum(n, 1e-8, out=n)
    return (Z / n).astype(np.float32)


def pca_reduce(Z, n_components=64, seed=0):
    """Reduce dense embeddings so that per-pair dot products stay cheap."""
    if Z.shape[1] <= n_components:
        return l2_normalise(Z)
    from sklearn.decomposition import PCA

    p = PCA(n_components=n_components, random_state=seed, copy=False)
    return l2_normalise(p.fit_transform(Z.astype(np.float32)))


# --------------------------------------------------------------------------
# 2. User-side profiles  (sparse user x article matrix @ dense article matrix)
# --------------------------------------------------------------------------


def user_article_matrix(u_idx, a_code, n_users, n_articles, normalise_rows=True):
    """
    Sparse (users x articles) incidence matrix from an exploded history.
    Multiplying it by any article-level matrix gives mean-pooled user profiles
    in one BLAS/SciPy call — no python loop over users.
    """
    data = np.ones(len(u_idx), dtype=np.float32)
    S = sp.csr_matrix((data, (u_idx, a_code)), shape=(n_users, n_articles))
    S.sum_duplicates()
    if normalise_rows:
        counts = np.asarray(S.sum(axis=1)).ravel()
        np.maximum(counts, 1.0, out=counts)
        S = sp.diags((1.0 / counts).astype(np.float32)) @ S
    return S.tocsr()


def profile_from(S, M):
    """Mean-pooled user profile in the space of M, L2-normalised."""
    return l2_normalise(np.asarray(S @ M, dtype=np.float32))


def rowwise_cosine(U, A, u_idx, a_code, block=1_000_000):
    """
    cos(u_i, a_i) for aligned index arrays, computed in blocks so the gathered
    (n_pairs x d) temporaries never exceed ~block*d*4 bytes.
    U and A must already be L2-normalised.
    """
    n = len(u_idx)
    out = np.empty(n, dtype=np.float32)
    for s in range(0, n, block):
        e = min(s + block, n)
        out[s:e] = np.einsum("ij,ij->i", U[u_idx[s:e]], A[a_code[s:e]], optimize=True)
    return out


# --------------------------------------------------------------------------
# 3. Ranking / beyond-accuracy metrics
# --------------------------------------------------------------------------


def _group_offsets(group_ids):
    """Start offsets of contiguous runs. Input must be sorted by group."""
    group_ids = np.asarray(group_ids)
    if len(group_ids) == 0:
        return np.array([0], dtype=np.int64)
    change = np.flatnonzero(group_ids[1:] != group_ids[:-1]) + 1
    return np.concatenate([[0], change, [len(group_ids)]]).astype(np.int64)


def group_auc(scores, labels):
    """Rank-based AUC for a single impression (Mann-Whitney U)."""
    n_pos = labels.sum()
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.nan
    order = np.argsort(scores, kind="stable")
    ranks = np.empty(len(scores), dtype=np.float64)
    s_sorted = scores[order]
    # average ranks for ties
    ranks[order] = np.arange(1, len(scores) + 1, dtype=np.float64)
    i = 0
    while i < len(s_sorted):
        j = i
        while j + 1 < len(s_sorted) and s_sorted[j + 1] == s_sorted[i]:
            j += 1
        if j > i:
            ranks[order[i : j + 1]] = (i + j + 2) / 2.0
        i = j + 1
    return (ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


def _dcg(rel):
    return float(np.sum(rel / np.log2(np.arange(2, len(rel) + 2))))


def group_ndcg(scores, labels, k):
    order = np.argsort(-scores, kind="stable")
    rel = labels[order][:k]
    ideal = np.sort(labels)[::-1][:k]
    idcg = _dcg(ideal)
    return _dcg(rel) / idcg if idcg > 0 else np.nan


def group_mrr(scores, labels):
    order = np.argsort(-scores, kind="stable")
    hits = np.flatnonzero(labels[order] == 1)
    return 1.0 / (hits[0] + 1) if len(hits) else 0.0


def evaluate_groups(
    group_ids,
    scores,
    labels,
    a_code=None,
    emb=None,
    pop_rate=None,
    topk_beyond=5,
    n_catalog=None,
):
    """
    Accuracy + beyond-accuracy metrics.

    Returns (per_group_dict_of_arrays, coverage_float). Keeping the *per-group*
    values (not just the means) is what makes bootstrap CIs and slicing free.
    """
    group_ids = np.asarray(group_ids)
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int8)
    off = _group_offsets(group_ids)
    n_groups = len(off) - 1

    out = {
        "auc": np.full(n_groups, np.nan),
        "mrr": np.full(n_groups, np.nan),
        "ndcg5": np.full(n_groups, np.nan),
        "ndcg10": np.full(n_groups, np.nan),
        "ild": np.full(n_groups, np.nan),
        "novelty": np.full(n_groups, np.nan),
    }
    covered = set()

    for g in range(n_groups):
        s, e = off[g], off[g + 1]
        sc, lb = scores[s:e], labels[s:e]
        out["auc"][g] = group_auc(sc, lb)
        out["mrr"][g] = group_mrr(sc, lb)
        out["ndcg5"][g] = group_ndcg(sc, lb, 5)
        out["ndcg10"][g] = group_ndcg(sc, lb, 10)

        if a_code is not None:
            top = np.argsort(-sc, kind="stable")[:topk_beyond]
            codes = np.asarray(a_code[s:e])[top]
            covered.update(codes.tolist())
            if emb is not None and len(codes) > 1:
                V = emb[codes]
                sim = V @ V.T
                m = len(codes)
                iu = np.triu_indices(m, 1)
                out["ild"][g] = float(1.0 - sim[iu].mean())
            if pop_rate is not None:
                p = np.clip(pop_rate[codes], 1e-9, None)
                out["novelty"][g] = float(np.mean(-np.log2(p)))

    coverage = (len(covered) / n_catalog) if (n_catalog and a_code is not None) else np.nan
    return out, coverage


def bootstrap_ci(values, n_boot=500, alpha=0.05, seed=0):
    """Percentile bootstrap over impressions (the unit of sampling)."""
    v = np.asarray(values, dtype=np.float64)
    v = v[~np.isnan(v)]
    if len(v) == 0:
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(v), size=(n_boot, len(v)))
    means = v[idx].mean(axis=1)
    return (float(v.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2)))


def summarise(per_group, coverage, name="model", n_boot=300):
    rows = []
    for m in ["auc", "mrr", "ndcg5", "ndcg10", "ild", "novelty"]:
        mean, lo, hi = bootstrap_ci(per_group[m], n_boot=n_boot)
        rows.append({"model": name, "metric": m, "mean": mean, "ci_lo": lo, "ci_hi": hi})
    rows.append({"model": name, "metric": "coverage@5", "mean": coverage, "ci_lo": np.nan, "ci_hi": np.nan})
    return rows


# --------------------------------------------------------------------------
# 4. Candidate-generation recall@K  (Q2 / Q3)
# --------------------------------------------------------------------------


def recall_at_k_sparse(Q, W, truth_lists, pool_mask=None, ks=(50, 100, 200), batch=128):
    """
    Q: (n_queries x vocab) sparse query weights, W: (n_docs x vocab) BM25 weights.
    truth_lists: list of arrays of ground-truth doc indices (article codes).
    """
    return _recall_generic(lambda qs: np.asarray((Q[qs] @ W.T).todense()), len(truth_lists), truth_lists, pool_mask, ks, batch)


def recall_at_k_dense(Uq, A, truth_lists, pool_mask=None, ks=(50, 100, 200), batch=256):
    """Brute-force ANN (exact) — Uq: (n_queries x d), A: (n_docs x d), both L2-normalised."""
    return _recall_generic(lambda qs: Uq[qs] @ A.T, len(truth_lists), truth_lists, pool_mask, ks, batch)


def _recall_generic(score_fn, n_q, truth_lists, pool_mask, ks, batch):
    kmax = max(ks)
    hits = {k: [] for k in ks}
    for s in range(0, n_q, batch):
        qs = np.arange(s, min(s + batch, n_q))
        S = np.asarray(score_fn(qs), dtype=np.float32)
        if pool_mask is not None:
            S[:, ~pool_mask] = -np.inf
        top = np.argpartition(-S, kth=min(kmax, S.shape[1] - 1), axis=1)[:, :kmax]
        row_scores = np.take_along_axis(S, top, axis=1)
        order = np.argsort(-row_scores, axis=1)
        top = np.take_along_axis(top, order, axis=1)
        for r, qi in enumerate(qs):
            truth = np.asarray(truth_lists[qi])
            if len(truth) == 0:
                continue
            for k in ks:
                inter = np.isin(truth, top[r, :k]).sum()
                hits[k].append(inter / len(truth))
    return {k: (float(np.mean(v)) if v else np.nan) for k, v in hits.items()}

## 1. Locating the data (Q1, step 1)

Kaggle mounts every attached dataset read-only under `/kaggle/input/<slug>/...` and the
folder depth depends on how the uploader zipped it. We therefore *search* for the files
rather than assuming a layout, and fail immediately with a readable message instead of
40 minutes later.

In [7]:
ROOT = Path(CFG["input_root"])


def find_one(pattern, must=True, prefer=None):
    hits = sorted(ROOT.rglob(pattern))
    hits = [h for h in hits if "__MACOSX" not in str(h)]
    if prefer:
        pref = [h for h in hits if prefer in str(h)]
        hits = pref or hits
    if not hits:
        if must:
            raise FileNotFoundError(f"could not find {pattern} under {ROOT}")
        return None
    return hits[0]


PATHS = {
    "articles": find_one("articles.parquet", prefer="testset") or find_one("articles.parquet"),
    "train_beh": find_one("train/behaviors.parquet"),
    "train_hist": find_one("train/history.parquet"),
    "val_beh": find_one("validation/behaviors.parquet"),
    "val_hist": find_one("validation/history.parquet"),
    "test_beh": find_one("test/behaviors.parquet"),
    "test_hist": find_one("test/history.parquet"),
}
# the test-period catalogue is a superset — merge both if two files exist
ARTICLE_FILES = sorted({str(p) for p in ROOT.rglob("articles.parquet") if "__MACOSX" not in str(p)})
for k, v in PATHS.items():
    print(f"{k:11s} {v}")
print("article files:", ARTICLE_FILES)

for k in ["train_beh", "val_beh", "test_beh"]:
    n = pq.ParquetFile(PATHS[k]).metadata.num_rows
    print(f"{k}: {n:,} rows")

articles    /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_testset/ebnerd_testset/articles.parquet
train_beh   /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/train/behaviors.parquet
train_hist  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/train/history.parquet
val_beh     /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/validation/behaviors.parquet
val_hist    /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/validation/history.parquet
test_beh    /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_testset/ebnerd_testset/test/behaviors.parquet
test_hist   /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_testset/ebnerd_testset/test/history.parquet
article files: ['/kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/articles.parquet', '/kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_testset/ebnerd_testset/articles.parquet']
train_beh: 12,063,890 rows
val_beh: 12,566,385 rows
test_beh: 13,536,71

## 2. Article catalogue + direct-address code map (Q1, step 2)

`article_id` is already an integer, so the code map is a plain array
`code_of_aid[article_id] -> row`. Every article attribute below is then a numpy array
indexed by `code`, which turns "look up the category of 150 M candidates" into one
fancy-index instead of a join.

EB-NeRD gives us things MIND does not, and they matter:
`published_time` (a *true* article age, not an inferred first-seen), `sentiment_score`,
`premium`, `article_type`, and the global `total_pageviews`. The last one is a
whole-corpus aggregate that includes the future, so it is tagged serving-unsafe and gets
ablated in §11 rather than quietly boosting the leaderboard score.

In [8]:
t0 = tic("article catalogue")
ART_COLS = ["article_id", "title", "subtitle", "category", "subcategory", "published_time",
            "premium", "article_type", "sentiment_score", "sentiment_label",
            "total_pageviews", "total_inviews"]


def read_articles(path):
    have = set(pq.ParquetFile(path).schema_arrow.names)
    return pl.read_parquet(path, columns=[c for c in ART_COLS if c in have])


articles = pl.concat([read_articles(p) for p in ARTICLE_FILES], how="diagonal_relaxed")
articles = articles.unique(subset=["article_id"], keep="first").sort("article_id")

for c, fill in [("title", ""), ("subtitle", ""), ("sentiment_score", 0.0),
                ("total_pageviews", 0.0), ("total_inviews", 0.0)]:
    if c in articles.columns:
        articles = articles.with_columns(pl.col(c).fill_null(fill))
if "premium" not in articles.columns:
    articles = articles.with_columns(premium=pl.lit(False))
if "article_type" not in articles.columns:
    articles = articles.with_columns(article_type=pl.lit("default"))

articles = articles.with_row_index("code").with_columns(pl.col("code").cast(pl.Int32))
N_ART = articles.height
MAX_AID = int(articles["article_id"].max())
code_of_aid = np.full(MAX_AID + 2, -1, dtype=np.int32)
code_of_aid[articles["article_id"].to_numpy()] = articles["code"].to_numpy()

cat_of_code = articles["category"].fill_null(-1).cast(pl.Int32).to_numpy().copy()
cat_of_code = cat_of_code - cat_of_code.min()
N_CAT = int(cat_of_code.max()) + 1
if "subcategory" in articles.columns:
    sub_of_code = (
        articles["subcategory"].list.first().fill_null(-1).cast(pl.Int32).to_numpy().copy()
        if articles["subcategory"].dtype == pl.List else
        articles["subcategory"].fill_null(-1).cast(pl.Int32).to_numpy().copy()
    )
else:
    sub_of_code = cat_of_code.copy()
sub_of_code = sub_of_code - sub_of_code.min()
N_SUB = int(sub_of_code.max()) + 1

pub_ts = (
    articles["published_time"].dt.epoch("s").cast(pl.Float64).to_numpy().astype(np.float64)
    if "published_time" in articles.columns else np.full(N_ART, np.nan)
)
pub_ts = np.where(np.isfinite(pub_ts), pub_ts, np.nan)
premium = articles["premium"].cast(pl.Int8).fill_null(0).to_numpy().astype(np.float32)
atype = articles["article_type"].cast(pl.Categorical).to_physical().cast(pl.Int32).to_numpy().astype(np.float32)
senti = articles["sentiment_score"].cast(pl.Float32).to_numpy().astype(np.float32)
pageviews = np.log1p(articles["total_pageviews"].cast(pl.Float64).fill_null(0).to_numpy()).astype(np.float32)
title_len = articles["title"].str.len_chars().to_numpy().astype(np.float32)

TITLES = articles["title"].to_list()
TEXTS = (articles["title"] + ". " + articles["subtitle"]).to_list()
toc(t0, f"{N_ART:,} articles | {N_CAT} categories | {N_SUB} subcategories")

[11:37:05] article catalogue
    ...125,541 articles | 2974 categories | 2970 subcategories 0.8s


## 3. Streaming I/O helpers

`pq.ParquetFile.iter_batches` gives bounded-memory reads with a knob we control, unlike
`read_parquet(...).slice()` (re-reads the file) or a lazy `group_by` over a 130 M-row
explode (the streaming engine will do it, but the peak is not something you can predict
from the notebook). Every heavy pass in this notebook is one of these loops.

In [9]:
def iter_parquet(path, columns=None, batch_rows=None):
    pf = pq.ParquetFile(path)
    have = set(pf.schema_arrow.names)
    cols = [c for c in columns if c in have] if columns else None
    for rb in pf.iter_batches(batch_size=batch_rows or CFG["batch_rows"], columns=cols):
        yield pl.from_arrow(rb)


BEH_COLS = ["impression_id", "user_id", "impression_time", "article_ids_inview",
            "article_ids_clicked", "device_type", "is_subscriber", "is_sso_user",
            "read_time", "scroll_percentage", "is_beyond_accuracy"]


def normalise_beh(df):
    if "article_ids_clicked" not in df.columns:
        df = df.with_columns(article_ids_clicked=pl.lit(None))
    for c, dflt in [("device_type", 0), ("is_subscriber", False), ("is_sso_user", False),
                    ("read_time", 0.0), ("scroll_percentage", 0.0)]:
        if c not in df.columns:
            df = df.with_columns(pl.lit(dflt).alias(c))
    return df.with_columns(
        ts=pl.col("impression_time").dt.epoch("s").cast(pl.Int64),
        device_type=pl.col("device_type").cast(pl.Int8, strict=False).fill_null(0),
        is_subscriber=pl.col("is_subscriber").cast(pl.Int8, strict=False).fill_null(0),
        is_sso_user=pl.col("is_sso_user").cast(pl.Int8, strict=False).fill_null(0),
        read_time=pl.col("read_time").cast(pl.Float32).fill_null(0.0),
        scroll_percentage=pl.col("scroll_percentage").cast(pl.Float32).fill_null(0.0),
    )


def load_behaviors(path, mod=None, limit=None, columns=BEH_COLS):
    """Streaming load with deterministic modulo subsampling (keeps the full time range)."""
    parts, n = [], 0
    for df in iter_parquet(path, columns):
        if mod:
            df = df.filter((pl.col("impression_id") % mod[0]) < mod[1])
        if df.height:
            parts.append(normalise_beh(df))
            n += df.height
        if limit and n >= limit * 3:
            break
    out = pl.concat(parts, how="diagonal_relaxed") if parts else pl.DataFrame()
    return out

## 4. Article-side feature store (Q1, step 4)

Same contract as the MIND notebook, recomputed **per split from that split's own files**:

| array | source | serving-safe? |
|---|---|---|
| `hist_rate` | that split's `history.parquet` (clicks strictly *before* the window) | yes |
| `inview_rate` | in-view counts across that split's behaviours | no — batch aggregate |
| `pseudo_ctr` | `hist_pop / (inview + 20)` | no |
| `first_seen` | earliest display timestamp | yes (streaming) |
| `age_h` | `impression_time − published_time` | yes |

No click label ever enters a feature. That is not a stylistic choice: the test file has
no `article_ids_clicked` column at all, so a click-derived feature would simply be
unreproducible at inference and the train/test feature distributions would diverge.

In [10]:
def article_stats(beh_path, hist_path, n_rows_hint=None):
    inview = np.zeros(N_ART, dtype=np.int64)
    first_seen = np.full(N_ART, np.inf, dtype=np.float64)
    n_imp = 0
    for df in iter_parquet(beh_path, ["impression_id", "impression_time", "article_ids_inview"]):
        n_imp += df.height
        d = (
            df.with_columns(ts=pl.col("impression_time").dt.epoch("s"))
            .select("article_ids_inview", "ts")
            .explode("article_ids_inview")
            .drop_nulls("article_ids_inview")
            .group_by("article_ids_inview")
            .agg(pl.len().alias("n"), pl.col("ts").min().alias("t0"))
        )
        aid = np.clip(d["article_ids_inview"].to_numpy().astype(np.int64), 0, MAX_AID)
        codes = code_of_aid[aid]
        ok = codes >= 0
        np.add.at(inview, codes[ok], d["n"].to_numpy()[ok])
        np.minimum.at(first_seen, codes[ok], d["t0"].to_numpy()[ok].astype(np.float64))

    hist_pop = np.zeros(N_ART, dtype=np.int64)
    for df in iter_parquet(hist_path, ["user_id", HIST_ART_COL]):
        d = (
            df.select(pl.col(HIST_ART_COL).list.tail(CFG["hist_member"]).alias("a"))
            .explode("a").drop_nulls("a")
            .group_by("a").agg(pl.len().alias("n"))
        )
        aid = np.clip(d["a"].to_numpy().astype(np.int64), 0, MAX_AID)
        codes = code_of_aid[aid]
        ok = codes >= 0
        np.add.at(hist_pop, codes[ok], d["n"].to_numpy()[ok])

    scale = 1e6 / max(n_imp, 1)
    return dict(
        hist_rate=(hist_pop * scale).astype(np.float32),
        inview_rate=(inview * scale).astype(np.float32),
        pseudo_ctr=(hist_pop / (inview + 20.0)).astype(np.float32),
        first_seen=first_seen,
        pop_prob=(inview / max(inview.sum(), 1)).astype(np.float64),
        pool_mask=(inview > 0),
        n_impressions=n_imp,
    )


# detect the history column name once (bundles differ: article_id_fixed vs article_ids)
_hs = set(pq.ParquetFile(PATHS["train_hist"]).schema_arrow.names)
HIST_ART_COL = next(c for c in ["article_id_fixed", "article_ids_fixed", "article_ids", "article_id"] if c in _hs)
print("history article column:", HIST_ART_COL, "|", sorted(_hs))

history article column: article_id_fixed | ['article_id_fixed', 'impression_time_fixed', 'read_time_fixed', 'scroll_percentage_fixed', 'user_id']


## 5. Text representations: lexical (TF-IDF → LSA) and semantic

Danish, so no English stop-word list and no English-pretrained encoder. Two spaces over
`title + subtitle` (the body is available but adds ~1.5 GB of text for a signal that is
largely duplicated by the headline in click prediction — a deliberate cut, revisited in
the design note):

* **lexical**: TF-IDF → 64-d truncated SVD.
* **semantic**: whichever is available, in order —
  1. the *provided* EB-NeRD artefacts (`Ekstra_Bladet_word2vec`, 300-d, or the
     multilingual BERT document vectors, 768-d), PCA-reduced to 64-d;
  2. `paraphrase-multilingual-MiniLM-L12-v2` encoded on the GPU;
  3. a character-n-gram LSA fallback, which for Danish compound nouns is a surprisingly
     decent substitute and never crashes the run.

Everything is reduced to 64-d for the same reason as in MIND: a per-pair cosine over
150 M pairs has to be a 64-wide `einsum` in blocks, not a 768-wide gather.

In [11]:
t0 = tic("TF-IDF + LSA")
tfidf_vec, TFIDF = build_tfidf(TEXTS, stop_words=None, min_df=CFG["tfidf_min_df"])
LSA = svd_reduce(TFIDF, CFG["dim"], seed=CFG["seed"])
toc(t0, f"vocab={TFIDF.shape[1]:,} → {LSA.shape}")


def load_provided_embeddings():
    """Any parquet with an article_id column and one list<float> column will do."""
    for p in sorted(ROOT.rglob("*.parquet")):
        name = p.name.lower()
        if not any(k in str(p).lower() for k in ["word2vec", "bert", "embedding", "contrastive", "roberta"]):
            continue
        try:
            schema = pq.ParquetFile(p).schema_arrow
            names = list(schema.names)
            if "article_id" not in names:
                continue
            vec_col = next((n for n in names if n != "article_id"), None)
            df = pl.read_parquet(p)
            E = np.zeros((N_ART, len(df[vec_col][0])), dtype=np.float32)
            codes = code_of_aid[np.clip(df["article_id"].to_numpy().astype(np.int64), 0, MAX_AID)]
            ok = codes >= 0
            E[codes[ok]] = np.asarray(df[vec_col].to_list(), dtype=np.float32)[ok]
            print(f"  using provided embeddings: {p.name} ({vec_col}, dim={E.shape[1]}, "
                  f"coverage={ok.mean():.1%})")
            return E, f"provided:{p.stem}"
        except Exception as e:  # noqa: BLE001
            print(f"  skipping {p.name}: {type(e).__name__}")
    return None, None


def semantic_embeddings():
    E, kind = load_provided_embeddings()
    if E is not None:
        return pca_reduce(E, CFG["dim"], seed=CFG["seed"]), kind
    try:
        import torch
        from sentence_transformers import SentenceTransformer

        dev = "cuda" if torch.cuda.is_available() else "cpu"
        if dev == "cpu" and len(TEXTS) > 20_000:
            raise RuntimeError("no GPU — skipping transformer encoder")
        m = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=dev)
        E = m.encode(TEXTS, batch_size=384, convert_to_numpy=True,
                     normalize_embeddings=True, show_progress_bar=True)
        del m
        if dev == "cuda":
            torch.cuda.empty_cache()
        return pca_reduce(E.astype(np.float32), CFG["dim"], seed=CFG["seed"]), "mmini-lm"
    except Exception as e:  # noqa: BLE001
        print(f"  encoder unavailable ({type(e).__name__}: {e}); char-ngram LSA fallback")
        from sklearn.feature_extraction.text import TfidfVectorizer

        v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=CFG["tfidf_min_df"],
                            max_features=200_000, dtype=np.float32)
        return svd_reduce(v.fit_transform(TEXTS), CFG["dim"], seed=CFG["seed"]), "char-lsa"


t0 = tic("semantic embeddings")
EMB, EMB_KIND = semantic_embeddings()
toc(t0, f"{EMB_KIND} {EMB.shape}")
gc.collect()

[11:37:06] TF-IDF + LSA
    ...vocab=43,594 → (125541, 64) 7.8s
[11:37:14] semantic embeddings
  using provided embeddings: document_vector.parquet (document_vector, dim=300, coverage=100.0%)
    ...provided:document_vector (125541, 64) 9.1s


30

## 6. Split context: user profiles from `history.parquet`

EB-NeRD keeps user history in a separate file (unlike MIND's inline strings), which is
strictly better for us: we build the profiles once per split straight from the history
file, streamed, and never touch the 12 M-row behaviours table to do it.

Profiles are built exactly as in MIND — a sparse (users × articles) incidence matrix
multiplied into each dense space — with two horizons (last 30 / last 5).

In [12]:
class SplitContext:
    def __init__(self, beh_path, hist_path, name, labeled):
        self.name, self.labeled = name, labeled
        t0 = tic(f"[{name}] article stats")
        self.stats = article_stats(beh_path, hist_path)
        toc(t0, f"{self.stats['n_impressions']:,} impressions")

        t0 = tic(f"[{name}] user profiles")
        uids, rows_all, rows_rec, rows_mem, hist_len = [], [], [], [], []
        offset = 0
        for df in iter_parquet(hist_path, ["user_id", HIST_ART_COL]):
            df = df.with_row_index("i").with_columns(pl.col("i").cast(pl.Int64) + offset)
            offset += df.height
            uids.append(df["user_id"].to_numpy().astype(np.int64))
            hist_len.append(df[HIST_ART_COL].list.len().to_numpy().astype(np.float32))
            for k, bucket in [(CFG["hist_len"], rows_all), (CFG["hist_recent"], rows_rec),
                              (CFG["hist_member"], rows_mem)]:
                d = (
                    df.select("i", pl.col(HIST_ART_COL).list.tail(k).alias("a"))
                    .explode("a").drop_nulls("a")
                )
                aid = np.clip(d["a"].to_numpy().astype(np.int64), 0, MAX_AID)
                c = code_of_aid[aid]
                ok = c >= 0
                bucket.append((d["i"].to_numpy().astype(np.int64)[ok], c[ok].astype(np.int64)))

        self.user_ids = np.concatenate(uids) if uids else np.zeros(0, np.int64)
        self.n_users = len(self.user_ids)
        self.hist_len_arr = np.concatenate(hist_len) if hist_len else np.zeros(0, np.float32)
        # direct-address user map (EB-NeRD user_id is a uint32)
        max_uid = int(self.user_ids.max()) if self.n_users else 0
        self.uidx_of_user = np.full(max_uid + 2, -1, dtype=np.int32)
        self.uidx_of_user[self.user_ids] = np.arange(self.n_users, dtype=np.int32)
        self.max_uid = max_uid

        def stack(bucket):
            if not bucket:
                return np.zeros(0, np.int64), np.zeros(0, np.int64)
            return np.concatenate([b[0] for b in bucket]), np.concatenate([b[1] for b in bucket])

        u_all, c_all = stack(rows_all)
        u_rec, c_rec = stack(rows_rec)
        u_mem, c_mem = stack(rows_mem)
        del rows_all, rows_rec, rows_mem
        gc.collect()

        S_all = user_article_matrix(u_all, c_all, self.n_users, N_ART)
        S_rec = user_article_matrix(u_rec, c_rec, self.n_users, N_ART)
        self.user_lsa = profile_from(S_all, LSA)
        self.user_emb = profile_from(S_all, EMB)
        self.user_lsa_r = profile_from(S_rec, LSA)
        self.user_emb_r = profile_from(S_rec, EMB)

        onehot_cat = sp.csr_matrix(
            (np.ones(N_ART, np.float32), (np.arange(N_ART), cat_of_code)), shape=(N_ART, N_CAT)
        )
        self.user_cat = np.asarray((S_all @ onehot_cat).todense(), dtype=np.float32)

        onehot_sub = sp.csr_matrix(
            (np.ones(N_ART, np.float32), (np.arange(N_ART), sub_of_code)), shape=(N_ART, N_SUB)
        )
        US = (S_all @ onehot_sub).tocsr()
        self.user_top_sub = np.full(self.n_users, -1, dtype=np.int32)
        for i in range(self.n_users):
            s, e = US.indptr[i], US.indptr[i + 1]
            if e > s:
                self.user_top_sub[i] = US.indices[s + int(np.argmax(US.data[s:e]))]

        self.hist_keys = np.unique(u_mem * np.int64(N_ART) + c_mem) if len(u_mem) else np.zeros(0, np.int64)
        del u_all, c_all, u_rec, c_rec, u_mem, c_mem, S_all, S_rec, US
        gc.collect()
        toc(t0, f"{self.n_users:,} users")

    def uidx(self, user_ids):
        u = np.clip(np.asarray(user_ids, dtype=np.int64), 0, self.max_uid)
        return np.maximum(self.uidx_of_user[u], 0), (self.uidx_of_user[u] >= 0)

    def free(self):
        for a in ["user_lsa", "user_emb", "user_lsa_r", "user_emb_r", "user_cat",
                  "hist_keys", "user_top_sub", "uidx_of_user"]:
            setattr(self, a, None)
        gc.collect()

### The pair featuriser

One function for train / validation / test. `labeled` only decides whether a `y` vector
is produced; every feature column is computed by the same code on the same inputs, which
is what makes train/serve skew structurally impossible rather than merely unlikely.

In [13]:
FEATURES = [
    "pos", "rel_pos", "n_inview",
    "hist_rate", "inview_rate", "pseudo_ctr", "pageviews",
    "age_h", "is_fresh", "seen_age_h",
    "lsa_cos", "lsa_cos_recent", "emb_cos", "emb_cos_recent",
    "cat_aff", "sub_top_match", "in_history",
    "hist_len", "known_user",
    "hour", "dow", "device_type", "is_subscriber", "is_sso_user",
    "ctx_read_time", "ctx_scroll",
    "premium", "atype", "sentiment", "title_len",
]
# v2: within-impression relative features
FEATURES += [
    "age_rel_rank", "pop_rel_rank", "hist_rel_rank",
    "emb_cos_c", "emb_cos_recent_c", "lsa_cos_c",
]
SERVING_UNSAFE = ["inview_rate", "pseudo_ctr", "pageviews"]

def _within_impression(grp, vals, descending=True):
    """Rank and mean-centre a feature *inside* each impression.

    Trees compare a feature against a global threshold and cannot express
    "freshest in THIS list". These two transforms make the comparison explicit.
    """
    df = pl.DataFrame({"g": grp, "v": np.asarray(vals, dtype=np.float32)})
    n = pl.len().over("g")
    out = df.select(
        rel_rank=((pl.col("v").rank("average", descending=descending).over("g") - 1.0)
                  / pl.when(n > 1).then(n - 1).otherwise(1)).cast(pl.Float32),
        centered=(pl.col("v") - pl.col("v").mean().over("g")).cast(pl.Float32),
    )
    return out["rel_rank"].to_numpy(), out["centered"].to_numpy()
    
def featurise(chunk, ctx):
    labeled = ctx.labeled
    keep = ["impression_id", "user_id", "ts", "impression_time", "article_ids_inview",
            "device_type", "is_subscriber", "is_sso_user", "read_time", "scroll_percentage"]
    if labeled:
        keep.append("article_ids_clicked")
    d = chunk.select(keep)
    d = d.with_columns(n_inview=pl.col("article_ids_inview").list.len())
    d = d.with_columns(pos=pl.int_ranges(0, pl.col("n_inview")))
    d = d.explode(["article_ids_inview", "pos"]).rename({"article_ids_inview": "article_id"})
    if labeled:
        d = d.with_columns(y=pl.col("article_ids_clicked").list.contains(pl.col("article_id")).cast(pl.Int8))
    d = d.with_columns(hour=pl.col("impression_time").dt.hour(),
                       dow=pl.col("impression_time").dt.weekday())

    aid = np.clip(d["article_id"].to_numpy().astype(np.int64), 0, MAX_AID)
    raw_code = code_of_aid[aid]
    known = (raw_code >= 0).astype(np.float32)
    code = np.maximum(raw_code, 0).astype(np.int64)
    uidx, known_user = ctx.uidx(d["user_id"].to_numpy())
    uidx = uidx.astype(np.int64)
    ts = d["ts"].to_numpy().astype(np.float64)
    pos = d["pos"].to_numpy().astype(np.float32)
    n_inview = d["n_inview"].to_numpy().astype(np.float32)

    st = ctx.stats
    n = len(code)
    X = np.empty((n, len(FEATURES)), dtype=np.float32)
    col = {f: i for i, f in enumerate(FEATURES)}

    X[:, col["pos"]] = pos
    X[:, col["rel_pos"]] = pos / np.maximum(n_inview, 1)
    X[:, col["n_inview"]] = n_inview
    X[:, col["hist_rate"]] = np.log1p(st["hist_rate"][code]) * known
    X[:, col["inview_rate"]] = np.log1p(st["inview_rate"][code]) * known
    X[:, col["pseudo_ctr"]] = st["pseudo_ctr"][code] * known
    X[:, col["pageviews"]] = pageviews[code] * known

    age = (ts - pub_ts[code]) / 3600.0
    X[:, col["age_h"]] = np.clip(np.nan_to_num(age, nan=-1.0), -1, 24 * 60)
    X[:, col["is_fresh"]] = (np.nan_to_num(age, nan=1e9) < 6).astype(np.float32)
    fs = st["first_seen"][code]
    X[:, col["seen_age_h"]] = np.clip(
        np.where(np.isfinite(fs), (ts - fs) / 3600.0, -1.0), -1, 24 * 60)

    ku = known * known_user.astype(np.float32)
    X[:, col["lsa_cos"]] = rowwise_cosine(ctx.user_lsa, LSA, uidx, code) * ku
    X[:, col["lsa_cos_recent"]] = rowwise_cosine(ctx.user_lsa_r, LSA, uidx, code) * ku
    X[:, col["emb_cos"]] = rowwise_cosine(ctx.user_emb, EMB, uidx, code) * ku
    X[:, col["emb_cos_recent"]] = rowwise_cosine(ctx.user_emb_r, EMB, uidx, code) * ku
    X[:, col["cat_aff"]] = ctx.user_cat[uidx, cat_of_code[code]] * ku
    X[:, col["sub_top_match"]] = (ctx.user_top_sub[uidx] == sub_of_code[code]).astype(np.float32) * ku

    if len(ctx.hist_keys):
        keys = uidx * np.int64(N_ART) + code
        ip = np.clip(np.searchsorted(ctx.hist_keys, keys), 0, len(ctx.hist_keys) - 1)
        X[:, col["in_history"]] = (ctx.hist_keys[ip] == keys).astype(np.float32) * ku
    else:
        X[:, col["in_history"]] = 0.0

    X[:, col["hist_len"]] = np.log1p(ctx.hist_len_arr[uidx] * known_user)
    X[:, col["known_user"]] = known_user.astype(np.float32)
    X[:, col["hour"]] = d["hour"].to_numpy().astype(np.float32)
    X[:, col["dow"]] = d["dow"].to_numpy().astype(np.float32)
    X[:, col["device_type"]] = d["device_type"].to_numpy().astype(np.float32)
    X[:, col["is_subscriber"]] = d["is_subscriber"].to_numpy().astype(np.float32)
    X[:, col["is_sso_user"]] = d["is_sso_user"].to_numpy().astype(np.float32)
    X[:, col["ctx_read_time"]] = np.log1p(np.clip(d["read_time"].to_numpy().astype(np.float32), 0, 1e5))
    X[:, col["ctx_scroll"]] = d["scroll_percentage"].to_numpy().astype(np.float32)
    X[:, col["premium"]] = premium[code] * known
    X[:, col["atype"]] = atype[code] * known
    X[:, col["sentiment"]] = senti[code] * known
    X[:, col["title_len"]] = title_len[code] * known
        # ---- within-impression relative features -------------------------------
    grp = np.cumsum(pos == 0) - 1          # rows of one impression are contiguous
    rr, cc = _within_impression(grp, X[:, col["age_h"]], descending=False)
    X[:, col["age_rel_rank"]] = rr
    rr, cc = _within_impression(grp, X[:, col["inview_rate"]], descending=True)
    X[:, col["pop_rel_rank"]] = rr
    rr, cc = _within_impression(grp, X[:, col["hist_rate"]], descending=True)
    X[:, col["hist_rel_rank"]] = rr
    rr, cc = _within_impression(grp, X[:, col["emb_cos"]], descending=True)
    X[:, col["emb_cos_c"]] = cc
    rr, cc = _within_impression(grp, X[:, col["emb_cos_recent"]], descending=True)
    X[:, col["emb_cos_recent_c"]] = cc
    rr, cc = _within_impression(grp, X[:, col["lsa_cos"]], descending=True)
    X[:, col["lsa_cos_c"]] = cc
    y = d["y"].to_numpy().astype(np.int8) if labeled else None
    return (d["impression_id"].to_numpy(), d["pos"].to_numpy().astype(np.int32), code, X, y)


def iter_slices(df, n):
    for s in range(0, df.height, n):
        yield df.slice(s, n)


def featurise_all(beh, ctx, rows=None):
    rows = rows or max(1, int(CFG["pair_budget"] / 12))
    outs = [featurise(ch, ctx) for ch in iter_slices(beh, rows)]
    return (
        np.concatenate([o[0] for o in outs]),
        np.concatenate([o[1] for o in outs]),
        np.concatenate([o[2] for o in outs]),
        np.vstack([o[3] for o in outs]),
        np.concatenate([o[4] for o in outs]) if outs[0][4] is not None else None,
    )

## 7. Temporal split (Q1, step 3)

* `ebnerd_large/train` → the first days fit the model, the **last day** is the
  early-stopping set. Never a random split.
* `ebnerd_large/validation` → a strictly later window, used only for the reported
  offline metrics, mirroring the train→test gap of the leaderboard.
* `ebnerd_testset/test` → the leaderboard.

Sampling is `impression_id % 25 < 2` (~8 %), which is deterministic, needs no shuffle,
and preserves the full time range — `head(n)` would silently give us the first hours of
a week and a model that has never seen a weekend.

In [14]:
t0 = tic("loading train behaviours (sampled)")
beh_train = load_behaviors(PATHS["train_beh"], mod=CFG["train_mod"])
toc(t0, f"{beh_train.height:,} impressions sampled")

t_max = beh_train["ts"].max()
SPLIT_TS = t_max - 24 * 3600
fit_pool = beh_train.filter(pl.col("ts") < SPLIT_TS)
es_pool = beh_train.filter(pl.col("ts") >= SPLIT_TS)
if es_pool.height == 0:
    fit_pool, es_pool = beh_train, beh_train


def subsample(df, n, seed=0):
    if df.height <= n:
        return df
    idx = np.random.default_rng(seed).choice(df.height, size=n, replace=False)
    return df[np.sort(idx)]


beh_fit = subsample(fit_pool, CFG["train_impressions"], 1)
beh_es = subsample(es_pool, CFG["es_impressions"], 2)
print("train window:", beh_train["impression_time"].min(), "→", beh_train["impression_time"].max())
print(f"fit={beh_fit.height:,}  early-stop={beh_es.height:,}")

t0 = tic("loading validation behaviours (sampled)")
beh_eval = subsample(
    load_behaviors(PATHS["val_beh"], mod=CFG["eval_mod"]), CFG["eval_impressions"], 3)
toc(t0, f"{beh_eval.height:,} impressions")

[11:37:23] loading train behaviours (sampled)
    ...1,929,201 impressions sampled 5.1s
train window: 2023-05-18 07:00:00 → 2023-05-25 06:59:59
fit=700,000  early-stop=120,000
[11:37:29] loading validation behaviours (sampled)
    ...80,000 impressions 5.5s


## 8. Q2/Q3 — BM25 and embedding candidate generation, recall@K

The retrieval study runs on the validation split. Query = titles of the reader's last 30
clicks; corpus = the full 125 k-article catalogue; ground truth = the articles they
actually clicked in that impression.

The `full` vs `live pool` contrast is the finding worth putting in the design note: in
news, an article that is three weeks old is *unclickable* no matter how well it matches
the query, so an index-freshness filter is worth more than any amount of query
engineering. The cold-start slice shows the mirror image — content retrieval is the only
tool that works when there is no behaviour to exploit.

In [15]:
ctx_eval = SplitContext(PATHS["val_beh"], PATHS["val_hist"], "validation", labeled=True)

[11:37:34] [validation] article stats
    ...12,566,385 impressions 10.7s
[11:37:45] [validation] user profiles
    ...791,582 users 18.2s


## 9. Ranking model

Identical modelling choice to the MIND notebook — LightGBM, binary log-loss, ~30 tabular
features — for identical reasons: the candidate list is given, the dominant signals are
tabular, AUC is the primary metric, and 150 M pairs have to be scored inside a free
session. The frozen text encoder enters as four cosine features rather than being
trained end-to-end; that is the accuracy we trade for a pipeline that finishes.

In [16]:
import lightgbm as lgb

ctx_train = SplitContext(PATHS["train_beh"], PATHS["train_hist"], "train", labeled=True)

t0 = tic("featurising fit/es")
_, _, _, X_fit, y_fit = featurise_all(beh_fit, ctx_train)
_, _, _, X_es, y_es = featurise_all(beh_es, ctx_train)
toc(t0, f"fit {X_fit.shape} (CTR {y_fit.mean():.4f}) | es {X_es.shape}")

params = dict(
    objective="binary", metric="auc", learning_rate=CFG["learning_rate"],
    num_leaves=CFG["num_leaves"], min_data_in_leaf=200, feature_fraction=0.9,
    bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, max_bin=127,
    num_threads=N_THREADS, verbose=-1, seed=CFG["seed"],
)


def train_model(feats, tag, rows=None):
    idx = [FEATURES.index(f) for f in feats]
    Xt = X_fit[:rows, idx] if rows else X_fit[:, idx]
    yt = y_fit[:rows] if rows else y_fit
    dtr = lgb.Dataset(Xt, label=yt, feature_name=feats, free_raw_data=True)
    dva = lgb.Dataset(X_es[:, idx], label=y_es, feature_name=feats,
                      reference=dtr, free_raw_data=True)
    m = lgb.train(params, dtr, num_boost_round=CFG["num_boost_round"], valid_sets=[dva],
                  valid_names=["es"],
                  callbacks=[lgb.early_stopping(60, verbose=False), lgb.log_evaluation(200)])
    print(f"{tag}: best_iter={m.best_iteration} es_auc={m.best_score['es']['auc']:.5f}")
    return m, idx


REL = ["age_rel_rank", "pop_rel_rank", "hist_rel_rank",
       "emb_cos_c", "emb_cos_recent_c", "lsa_cos_c"]
BASE = [f for f in FEATURES if f not in REL]
half = len(y_fit) // 2

model_A, IDX_A = train_model(BASE, "A base/small", rows=half)
model_B, IDX_B = train_model(BASE, "B base/big")
model,   IDX_FULL = train_model(FEATURES, "C +relative/big")
SAFE_FEATURES = [f for f in FEATURES if f not in SERVING_UNSAFE]
model_safe, IDX_SAFE = train_model(SAFE_FEATURES, "serving-safe")



del X_fit, y_fit, X_es, y_es
ctx_train.free()
del ctx_train
gc.collect()

[11:38:06] [train] article stats
    ...12,063,890 impressions 9.9s
[11:38:16] [train] user profiles
    ...788,090 users 17.9s
[11:38:34] featurising fit/es
    ...fit (7785484, 36) (CTR 0.0902) | es (1312493, 36) 27.6s
[200]	es's auc: 0.804128
A base/small: best_iter=320 es_auc=0.80509
[200]	es's auc: 0.8039
B base/big: best_iter=247 es_auc=0.80445
[200]	es's auc: 0.826889
[400]	es's auc: 0.829025
[600]	es's auc: 0.829766
C +relative/big: best_iter=572 es_auc=0.82988
[200]	es's auc: 0.821586
[400]	es's auc: 0.822599
serving-safe: best_iter=345 es_auc=0.82285


0

## 10. Q4 — offline evaluation harness

On the held-out validation window, with per-impression metrics, bootstrap CIs over
impressions, beyond-accuracy on the top-5, and two slices. The comparison set includes
the trivial baselines on purpose: if "rank by exposure" or "keep the publisher's own
display order" is within noise of the model, the model is not doing anything.

In [17]:
imp_ev, pos_ev, code_ev, X_ev, y_ev = featurise_all(beh_eval, ctx_eval)
order = np.lexsort((pos_ev, imp_ev))
imp_ev, pos_ev, code_ev, X_ev, y_ev = (a[order] for a in (imp_ev, pos_ev, code_ev, X_ev, y_ev))

scores = {
    "A base/small":    model_A.predict(X_ev[:, IDX_A], num_iteration=model_A.best_iteration),
    "B base/big":      model_B.predict(X_ev[:, IDX_B], num_iteration=model_B.best_iteration),
    "C +relative/big": model.predict(X_ev[:, IDX_FULL], num_iteration=model.best_iteration),
    "GBDT (serving-safe)": model_safe.predict(X_ev[:, IDX_SAFE], num_iteration=model_safe.best_iteration),
    "popularity only":  X_ev[:, FEATURES.index("inview_rate")].astype(np.float64),
    "freshness only":  -X_ev[:, FEATURES.index("age_h")].astype(np.float64),
    "display order":   -X_ev[:, FEATURES.index("pos")].astype(np.float64),
}

n_pool = int(ctx_eval.stats["pool_mask"].sum())
rows, per_group_store = [], {}
for name, sc in scores.items():
    pg, cov = evaluate_groups(imp_ev, sc, y_ev, a_code=code_ev, emb=EMB,
                              pop_rate=ctx_eval.stats["pop_prob"], topk_beyond=5, n_catalog=n_pool)
    per_group_store[name] = pg
    rows += summarise(pg, cov, name=name)

results = pl.DataFrame(rows)
print(results.filter(pl.col("metric").is_in(["auc", "mrr", "ndcg5", "ndcg10"])))
print(results.filter(pl.col("metric").is_in(["ild", "novelty", "coverage@5"])))
results.write_csv(Path(CFG["out_dir"]) / "ebnerd_eval_results.csv")

shape: (28, 5)
┌────────────────┬────────┬──────────┬──────────┬──────────┐
│ model          ┆ metric ┆ mean     ┆ ci_lo    ┆ ci_hi    │
│ ---            ┆ ---    ┆ ---      ┆ ---      ┆ ---      │
│ str            ┆ str    ┆ f64      ┆ f64      ┆ f64      │
╞════════════════╪════════╪══════════╪══════════╪══════════╡
│ A base/small   ┆ auc    ┆ 0.738044 ┆ 0.73626  ┆ 0.739901 │
│ A base/small   ┆ mrr    ┆ 0.505722 ┆ 0.503605 ┆ 0.508026 │
│ A base/small   ┆ ndcg5  ┆ 0.569826 ┆ 0.567722 ┆ 0.572062 │
│ A base/small   ┆ ndcg10 ┆ 0.609881 ┆ 0.608095 ┆ 0.611766 │
│ B base/big     ┆ auc    ┆ 0.737845 ┆ 0.736085 ┆ 0.739694 │
│ …              ┆ …      ┆ …        ┆ …        ┆ …        │
│ freshness only ┆ ndcg10 ┆ 0.420528 ┆ 0.418387 ┆ 0.422436 │
│ display order  ┆ auc    ┆ 0.500401 ┆ 0.497963 ┆ 0.502688 │
│ display order  ┆ mrr    ┆ 0.31391  ┆ 0.31198  ┆ 0.315785 │
│ display order  ┆ ndcg5  ┆ 0.345937 ┆ 0.343676 ┆ 0.348161 │
│ display order  ┆ ndcg10 ┆ 0.430913 ┆ 0.429011 ┆ 0.432829 │
└────────

In [18]:
import polars as pl
rows = []
for n in ["A base/small", "B base/big", "C +relative/big"]:
    m, lo, hi = bootstrap_ci(per_group_store[n]["auc"], n_boot=500)
    rows.append({"variant": n, "auc": m, "ci_lo": lo, "ci_hi": hi})
tbl = pl.DataFrame(rows); print(tbl)
tbl.write_csv("/kaggle/working/ebnerd_iteration_table.csv")

shape: (3, 4)
┌─────────────────┬──────────┬──────────┬──────────┐
│ variant         ┆ auc      ┆ ci_lo    ┆ ci_hi    │
│ ---             ┆ ---      ┆ ---      ┆ ---      │
│ str             ┆ f64      ┆ f64      ┆ f64      │
╞═════════════════╪══════════╪══════════╪══════════╡
│ A base/small    ┆ 0.738044 ┆ 0.736347 ┆ 0.739917 │
│ B base/big      ┆ 0.737845 ┆ 0.736078 ┆ 0.739712 │
│ C +relative/big ┆ 0.756486 ┆ 0.754819 ┆ 0.75842  │
└─────────────────┴──────────┴──────────┴──────────┘


In [19]:
uniq_imp = np.unique(imp_ev)
uidx_ev, known_ev = ctx_eval.uidx(beh_eval["user_id"].to_numpy())
imp2hist = dict(zip(beh_eval["impression_id"].to_list(),
                    (ctx_eval.hist_len_arr[uidx_ev] * known_ev).tolist()))
g_hist = np.array([imp2hist.get(i, 0.0) for i in uniq_imp], dtype=np.float32)

inview = ctx_eval.stats["inview_rate"]
thr = np.quantile(inview[inview > 0], 0.8) if (inview > 0).any() else 0.0
off = _group_offsets(imp_ev)
g_head = np.zeros(len(uniq_imp), dtype=bool)
for g in range(len(uniq_imp)):
    s, e = off[g], off[g + 1]
    cc = code_ev[s:e][y_ev[s:e] == 1]
    g_head[g] = bool(len(cc)) and bool(inview[cc].max() >= thr)

cold_thr = np.quantile(g_hist, 0.25)
slice_rows = []
for name, pg in per_group_store.items():
    for slabel, mask in [(f"cold users (<={cold_thr:.0f} clicks)", g_hist <= cold_thr),
                         ("warm users", g_hist > cold_thr),
                         ("head clicks", g_head), ("tail clicks", ~g_head)]:
        if mask.sum() < 20:
            continue
        m, lo, hi = bootstrap_ci(pg["auc"][mask], n_boot=200)
        slice_rows.append({"model": name, "slice": slabel, "n": int(mask.sum()),
                           "auc": m, "ci_lo": lo, "ci_hi": hi})
slice_df = pl.DataFrame(slice_rows)
print(slice_df)
slice_df.write_csv(Path(CFG["out_dir"]) / "ebnerd_eval_slices.csv")

shape: (28, 6)
┌────────────────┬──────────────────────────┬───────┬──────────┬──────────┬──────────┐
│ model          ┆ slice                    ┆ n     ┆ auc      ┆ ci_lo    ┆ ci_hi    │
│ ---            ┆ ---                      ┆ ---   ┆ ---      ┆ ---      ┆ ---      │
│ str            ┆ str                      ┆ i64   ┆ f64      ┆ f64      ┆ f64      │
╞════════════════╪══════════════════════════╪═══════╪══════════╪══════════╪══════════╡
│ A base/small   ┆ cold users (<=89 clicks) ┆ 20023 ┆ 0.741069 ┆ 0.737299 ┆ 0.744308 │
│ A base/small   ┆ warm users               ┆ 59977 ┆ 0.737034 ┆ 0.735069 ┆ 0.739239 │
│ A base/small   ┆ head clicks              ┆ 79816 ┆ 0.738584 ┆ 0.737083 ┆ 0.74062  │
│ A base/small   ┆ tail clicks              ┆ 184   ┆ 0.50368  ┆ 0.462687 ┆ 0.542947 │
│ B base/big     ┆ cold users (<=89 clicks) ┆ 20023 ┆ 0.74064  ┆ 0.736436 ┆ 0.744284 │
│ …              ┆ …                        ┆ …     ┆ …        ┆ …        ┆ …        │
│ freshness only ┆ tail clic

## 11. Q9 — leakage tests and the serving-safe ablation

The `article_ids_clicked` column must never reach a feature; the fit window must not
overlap the early-stopping window; and the featuriser must produce identical columns
whether or not labels are present in its input (the test file's schema literally lacks
the label column, so this is the exact code path inference will take).

In [20]:
def test_temporal_boundary():
    assert beh_fit["ts"].max() <= beh_es["ts"].min() or fit_pool is beh_train


def test_no_label_features():
    assert not ({"y", "label", "clicked", "article_ids_clicked"} & set(FEATURES))


def test_featuriser_is_label_agnostic():
    ch = beh_eval.head(64)
    _, _, c1, X1, _ = featurise(ch, ctx_eval)

    import copy
    fake = copy.copy(ctx_eval)
    fake.labeled = False
    _, _, c2, X2, y2 = featurise(ch.drop("article_ids_clicked"), fake)
    assert y2 is None and np.array_equal(c1, c2) and np.allclose(X1, X2, atol=1e-6)


def test_no_future_articles():
    """No candidate may be published after the impression that displays it."""
    ch = beh_eval.head(2000)
    _, _, code, X, _ = featurise(ch, ctx_eval)
    age = X[:, FEATURES.index("age_h")]
    bad = float((age < -1e-6).mean())
    assert bad < 0.02, f"{bad:.2%} of candidates look published after their impression"


for fn in [test_temporal_boundary, test_no_label_features,
           test_featuriser_is_label_agnostic, test_no_future_articles]:
    fn()
    print(f"PASS {fn.__name__}")

ablation = pl.DataFrame([
    {"model": "all features (incl. batch aggregates)",
     "dev_auc": float(np.nanmean(per_group_store["C +relative/big"]["auc"]))},
    {"model": "serving-safe only",
     "dev_auc": float(np.nanmean(per_group_store["GBDT (serving-safe)"]["auc"]))},
])
print(ablation)
ablation.write_csv(Path(CFG["out_dir"]) / "ebnerd_ablation.csv")

ctx_eval.free()
del X_ev, imp_ev, pos_ev, code_ev, y_ev
gc.collect()

PASS test_temporal_boundary
PASS test_no_label_features
PASS test_featuriser_is_label_agnostic
PASS test_no_future_articles
shape: (2, 2)
┌─────────────────────────────────┬──────────┐
│ model                           ┆ dev_auc  │
│ ---                             ┆ ---      │
│ str                             ┆ f64      │
╞═════════════════════════════════╪══════════╡
│ all features (incl. batch aggr… ┆ 0.756486 │
│ serving-safe only               ┆ 0.741816 │
└─────────────────────────────────┴──────────┘


0

## 12. Q5 — test inference and `predictions.zip`

13.5 M impressions; the beyond-accuracy rows carry 250 candidates each, so chunking by
*rows* would produce wildly uneven memory. We chunk by **pair budget** instead:
accumulate parquet batches until the cumulative in-view count crosses 1.5 M, then flush.
Peak memory is therefore flat regardless of which part of the file we are in.

One line per behaviour row, ranks in the original `article_ids_inview` order, rank 1 =
most likely click — the format the RecSys'24 evaluator expects.

In [21]:
t0 = tic("test context")
ctx_test = SplitContext(PATHS["test_beh"], PATHS["test_hist"], "test", labeled=False)
toc(t0)

PRED_TXT = Path(CFG["out_dir"]) / "predictions.txt"
PRED_ZIP = Path(CFG["out_dir"]) / "predictions.zip"
N_TEST_ROWS = pq.ParquetFile(PATHS["test_beh"]).metadata.num_rows


def pair_budget_chunks(path, budget):
    """Yield polars frames whose cumulative candidate count stays under `budget`."""
    buf, n_pairs = [], 0
    for df in iter_parquet(path, [c for c in BEH_COLS if c != "article_ids_clicked"]):
        df = normalise_beh(df)
        sizes = df["article_ids_inview"].list.len().to_numpy()
        start = 0
        run = 0
        for i, s in enumerate(sizes):
            run += int(s)
            if n_pairs + run >= budget:
                buf.append(df.slice(start, i - start + 1))
                yield pl.concat(buf, how="diagonal_relaxed")
                buf, n_pairs, run, start = [], 0, 0, i + 1
        if start < df.height:
            buf.append(df.slice(start, df.height - start))
            n_pairs += run
    if buf:
        yield pl.concat(buf, how="diagonal_relaxed")


n_written = 0
t0 = tic("scoring test set")
with open(PRED_TXT, "w") as fh:
    for ci, ch in enumerate(pair_budget_chunks(PATHS["test_beh"], CFG["pair_budget"])):
        imp, pos, code, X, _ = featurise(ch, ctx_test)
        s = model.predict(X[:, IDX_FULL], num_iteration=model.best_iteration)
        out = (
            pl.DataFrame({"row": np.repeat(np.arange(ch.height, dtype=np.int64) + n_written,
                                           ch["article_ids_inview"].list.len().to_numpy()),
                          "impression_id": imp, "pos": pos, "score": s})
            .sort(["row", "pos"])
            .with_columns(rank=pl.col("score").rank(method="ordinal", descending=True)
                          .over("row").cast(pl.Int32))
            .group_by("row", maintain_order=True)
            .agg(pl.col("impression_id").first(), pl.col("rank"))
            .with_columns(line=pl.format("{} [{}]", pl.col("impression_id"),
                                         pl.col("rank").cast(pl.List(pl.Utf8)).list.join(",")))
        )
        fh.write("\n".join(out["line"].to_list()) + "\n")
        n_written += ch.height
        del imp, pos, code, X, s, out, ch
        if ci % 20 == 0:
            gc.collect()
            print(f"  chunk {ci}: {n_written:,}/{N_TEST_ROWS:,} impressions", flush=True)
toc(t0, f"{n_written:,} rows")
assert n_written == N_TEST_ROWS, f"wrote {n_written} lines, expected {N_TEST_ROWS}"

with zipfile.ZipFile(PRED_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PRED_TXT, arcname="predictions.txt")
print(f"{PRED_ZIP} = {PRED_ZIP.stat().st_size/1e6:.1f} MB (txt {PRED_TXT.stat().st_size/1e6:.1f} MB)")

[11:58:09] test context
[11:58:09] [test] article stats
    ...13,536,710 impressions 16.8s
[11:58:26] [test] user profiles
    ...807,677 users 18.9s
    ... 35.8s
[11:58:45] scoring test set
  chunk 0: 127,452/13,536,710 impressions
  chunk 20: 2,693,566/13,536,710 impressions
  chunk 40: 5,264,622/13,536,710 impressions
  chunk 60: 7,830,855/13,536,710 impressions
  chunk 80: 10,398,161/13,536,710 impressions
  chunk 100: 12,955,278/13,536,710 impressions
  chunk 120: 13,439,011/13,536,710 impressions
    ...13,536,710 rows 5459.3s
/kaggle/working/predictions.zip = 230.1 MB (txt 703.1 MB)


### Submission validation

Streams the file back and checks every line against the in-view length of the
corresponding behaviour row: right number of lines, ranks a permutation of `1..N`.
Note that grouping is by **row**, not by `impression_id` — EB-NeRD's test file is not
guaranteed to have unique impression ids across the beyond-accuracy block, and the
evaluator consumes it row-wise.

In [22]:
def validate_submission(txt_path, beh_path):
    sizes = []
    for df in iter_parquet(beh_path, ["article_ids_inview"]):
        sizes.append(df["article_ids_inview"].list.len().to_numpy())
    sizes = np.concatenate(sizes)
    bad = seen = 0
    with open(txt_path) as fh:
        for i, line in enumerate(fh):
            _, ranks = line.split(" ", 1)
            r = [int(x) for x in ranks.strip()[1:-1].split(",")]
            n = int(sizes[i])
            if len(r) != n or sorted(r) != list(range(1, n + 1)):
                bad += 1
                if bad < 5:
                    print("  bad line", i, line[:100])
            seen += 1
    print(f"validated {seen:,} lines against {len(sizes):,} rows, {bad} malformed")
    assert bad == 0 and seen == len(sizes)


validate_submission(PRED_TXT, PATHS["test_beh"])
print("\nUpload predictions.zip → https://www.codabench.org/competitions/2469/")

validated 13,536,710 lines against 13,536,710 rows, 0 malformed

Upload predictions.zip → https://www.codabench.org/competitions/2469/


In [23]:
# def validate_submission(txt_path, beh_path):
#     sizes = []
#     for df in iter_parquet(beh_path, ["article_ids_inview"]):
#         sizes.append(df["article_ids_inview"].list.len().to_numpy())
#     sizes = np.concatenate(sizes)
#     bad = seen = 0
#     with open(txt_path) as fh:
#         for i, line in enumerate(fh):
#             _, ranks = line.split(" ", 1)
#             r = [int(x) for x in ranks.strip()[1:-1].split(",")]
#             n = int(sizes[i])
#             if len(r) != n or sorted(r) != list(range(1, n + 1)):
#                 bad += 1
#                 if bad < 5:
#                     print("  bad line", i, line[:100])
#             seen += 1
#     print(f"validated {seen:,} lines against {len(sizes):,} rows, {bad} malformed")
#     assert bad == 0 and seen == len(sizes)


# validate_submission(PRED_TXT, PATHS["test_beh"])
# print("\nUpload predictions.zip → https://www.codabench.org/competitions/2469/")

## 13. Where this breaks at 10× (for the design note)

* **User state, again.** `SplitContext` holds four dense `n_users × 64` matrices plus a
  `n_users × n_cat` affinity table — ~1 GB at 800 k users, ~10 GB at 8 M. This is the
  first thing to fail. Fix: compute profiles per chunk from the history rows of only the
  users in that chunk (a hash join per chunk), or precompute them once into a
  memory-mapped array / feature store keyed by user.
* **`np.unique` over the `in_history` keys** sorts 40 M int64 today, 400 M at 10× (~3 GB
  plus the sort). Fix: a per-chunk join, or a Bloom filter with a tolerated false-positive
  rate — the feature is binary and mildly noisy anyway.
* **The test pass is embarrassingly parallel and we run it serially.** 150 M pairs → 1.5 B.
  Fix: shard the parquet row-groups across Dask workers (or several Kaggle sessions) and
  concatenate the prediction files; nothing in the scoring path is stateful.
* **Article-side is fine.** 125 k articles → 1.25 M is still a 300 MB matrix. Text
  representation is not the bottleneck; the joins and the user state are.
* **What would actually change the answer at 10×**, rather than merely survive it: an
  incremental, streaming popularity counter (removes the batch-aggregate leakage *and*
  the extra pass) and a two-stage retrieve-then-rank architecture with an IVF index, so the ranker
  sees a few hundred candidates instead of every in-view article.

In [24]:
summary = {
    "version": "v2 (+within-impression relative features, 16% train sample)",
    "articles": N_ART,
    "fit_impressions": beh_fit.height,
    "eval_impressions": beh_eval.height,
    "test_rows": int(N_TEST_ROWS),
    "embedding": EMB_KIND,
    "best_iteration": int(model.best_iteration or 0),
    "es_auc": float(model.best_score["es"]["auc"]),
    "val_auc_A_base_small": float(np.nanmean(per_group_store["A base/small"]["auc"])),
    "val_auc_B_base_big": float(np.nanmean(per_group_store["B base/big"]["auc"])),
    "val_auc_C_relative": float(np.nanmean(per_group_store["C +relative/big"]["auc"])),
    "val_auc_safe": float(np.nanmean(per_group_store["GBDT (serving-safe)"]["auc"])),
}
print(json.dumps(summary, indent=2))
with open(Path(CFG["out_dir"]) / "ebnerd_summary_v2.json", "w") as f:
    json.dump(summary, f, indent=2)

{
  "version": "v2 (+within-impression relative features, 16% train sample)",
  "articles": 125541,
  "fit_impressions": 700000,
  "eval_impressions": 80000,
  "test_rows": 13536710,
  "embedding": "provided:document_vector",
  "best_iteration": 572,
  "es_auc": 0.8298751178634846,
  "val_auc_A_base_small": 0.7380441252452192,
  "val_auc_B_base_big": 0.737845339833093,
  "val_auc_C_relative": 0.7564863479224794,
  "val_auc_safe": 0.7418162849074103
}


In [25]:
import os
from pathlib import Path
from IPython.display import FileLink

W = Path("/kaggle/working")
for p in sorted(W.glob("*")):
    print(f"{p.stat().st_size/1e6:9.1f} MB  {p.name}")

# free up the working dir — the .txt is ~4x the zip and you don't need it
txt = W / "predictions.txt"
if txt.exists():
    txt.unlink()
    print("removed predictions.txt")

FileLink("predictions.zip")   # relative path, not absolute — absolute renders dead

      0.0 MB  .virtual_documents
      0.0 MB  ebnerd_ablation.csv
      0.0 MB  ebnerd_eval_results.csv
      0.0 MB  ebnerd_eval_slices.csv
      0.0 MB  ebnerd_iteration_table.csv
      0.0 MB  ebnerd_summary_v2.json
    703.1 MB  predictions.txt
    230.1 MB  predictions.zip
removed predictions.txt


/kaggle/working/predictions.zip

In [26]:
from pathlib import Path
for p in sorted(Path("/kaggle/input").rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size/1e6:9.1f} MB  {p}")
print("---- working ----")
for p in sorted(Path("/kaggle/working").rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size/1e6:9.1f} MB  {p}")

    150.8 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/articles.parquet
    541.9 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/train/behaviors.parquet
   1240.5 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/train/history.parquet
    586.5 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/validation/behaviors.parquet
   1125.3 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/validation/history.parquet
    150.8 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_testset/ebnerd_testset/articles.parquet
    567.8 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_testset/ebnerd_testset/test/behaviors.parquet
   1157.7 MB  /kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_testset/ebnerd_testset/test/history.parquet
    153.6 MB  /kaggle/input/datasets/sherleysonali/ebnerd-word2vec/Ekstra_Bladet_word2vec/document_vector.parquet
    291.3 MB  /kaggle/input/datasets/sherleyson

In [28]:
ctx_eval = SplitContext(PATHS["val_beh"], PATHS["val_hist"], "validation", labeled=True)

[14:00:50] [validation] article stats
    ...12,566,385 impressions 8.8s
[14:00:59] [validation] user profiles
    ...791,582 users 18.3s


In [29]:
# ============================================================================
# ENGINEERING BENCHMARKS (EB-NeRD) — paste at the END of the v2 notebook and run.
# Requires (already in memory after cells 1-11): TEXTS, EMB, LSA, ctx_eval,
#   beh_eval, PATHS, HIST_ART_COL, iter_parquet, code_of_aid, MAX_AID,
#   FEATURES, featurise, build_bm25, recall_at_k_sparse,
#   recall_at_k_dense, CFG, N_ART.
# Produces: engineering_benchmarks.csv  + printed tables for the design note.
# ============================================================================
import gc
import os
import time

import numpy as np
import polars as pl
import scipy.sparse as sp

try:
    import faiss
except ImportError:
    os.system("pip install -q faiss-cpu")
    import faiss

try:
    import psutil
    _PROC = psutil.Process()
    def rss_mb():
        return _PROC.memory_info().rss / 1e6
except ImportError:
    import resource
    def rss_mb():
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024


def timeit(fn, repeat=3):
    """Median wall-clock over `repeat` runs, plus the result of the last run."""
    ts, out = [], None
    for _ in range(repeat):
        gc.collect()
        t0 = time.perf_counter()
        out = fn()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts)), out


BENCH = []
def record(exp, **kw):
    BENCH.append({"experiment": exp, **kw})
    print(f"  {exp}: " + "  ".join(f"{k}={v}" for k, v in kw.items()), flush=True)


print("=" * 70)
print("EXPERIMENT 1 — retrieval index: build cost vs query latency vs recall")
print("=" * 70)

# --- rebuild the query set (the v2 notebook skipped the recall cells) --------
d = beh_eval.filter(pl.col("article_ids_clicked").list.len() > 0).head(3000)
uidx_all, known_all = ctx_eval.uidx(d["user_id"].to_numpy())
hist_txt = {}
wanted = set(int(x) for x in d["user_id"].to_numpy()[known_all])
for df in iter_parquet(PATHS["val_hist"], ["user_id", HIST_ART_COL]):
    df = df.filter(pl.col("user_id").is_in(list(wanted)))
    for u, arts in zip(df["user_id"].to_list(), df[HIST_ART_COL].to_list()):
        codes = [code_of_aid[a] for a in (arts or [])[-CFG["hist_len"]:] if 0 <= a <= MAX_AID]
        hist_txt[u] = " ".join(TITLES[c] for c in codes if c >= 0)
q_texts, q_truth, keep = [], [], []
for i, (u, clicked, ok) in enumerate(zip(d["user_id"].to_list(),
                                         d["article_ids_clicked"].to_list(), known_all.tolist())):
    if not ok or not hist_txt.get(u):
        continue
    t = np.array([code_of_aid[a] for a in (clicked or []) if 0 <= a <= MAX_AID], dtype=np.int64)
    t = t[t >= 0]
    if len(t) == 0:
        continue
    q_texts.append(hist_txt[u]); q_truth.append(t); keep.append(i)
q_uidx = uidx_all[keep]
Uq = np.ascontiguousarray(ctx_eval.user_emb[q_uidx])
print(f"query set: {len(q_texts):,} impressions | catalogue {N_ART:,} × {EMB.shape[1]}d")

KS = (50, 100, 200)
A = np.ascontiguousarray(EMB)

# --- 1a. BM25 sparse inverted index -----------------------------------------
t_build, (bm_vec, BM_W, BM_IDF) = timeit(
    lambda: build_bm25(TEXTS, stop_words=None, min_df=CFG["tfidf_min_df"]), repeat=1)
Q = bm_vec.transform(q_texts).astype(np.float32)
Q.data = np.minimum(Q.data, 5.0)
Q = Q.multiply(sp.csr_matrix(BM_IDF[np.newaxis, :])).tocsr()
idx_mb = (BM_W.data.nbytes + BM_W.indices.nbytes + BM_W.indptr.nbytes) / 1e6

lat = []
for s in range(0, min(512, Q.shape[0]), 32):
    t0 = time.perf_counter()
    S = np.asarray((Q[s:s + 32] @ BM_W.T).todense())
    np.argpartition(-S, 100, axis=1)[:, :100]
    lat.append((time.perf_counter() - t0) / 32 * 1000)
rec = recall_at_k_sparse(Q, BM_W, q_truth, None, ks=(100,))[100]
record("1_index", index="BM25 sparse CSR", build_s=round(t_build, 2), index_mb=round(idx_mb, 1),
       ms_per_query_p50=round(float(np.median(lat)), 3),
       ms_per_query_p99=round(float(np.percentile(lat, 99)), 3), recall_at_100=round(rec, 5))

# --- 1b. exact dense brute force (numpy) ------------------------------------
lat = []
for s in range(0, 512, 32):
    t0 = time.perf_counter()
    S = Uq[s:s + 32] @ A.T
    np.argpartition(-S, 100, axis=1)[:, :100]
    lat.append((time.perf_counter() - t0) / 32 * 1000)
rec = recall_at_k_dense(Uq, A, q_truth, None, ks=(100,))[100]
record("1_index", index="numpy brute force", build_s=0.0, index_mb=round(A.nbytes / 1e6, 1),
       ms_per_query_p50=round(float(np.median(lat)), 3),
       ms_per_query_p99=round(float(np.percentile(lat, 99)), 3), recall_at_100=round(rec, 5))

# --- 1c/1d. FAISS flat and IVF ----------------------------------------------
def faiss_bench(name, factory, nprobe=None):
    t0 = time.perf_counter()
    index = faiss.index_factory(A.shape[1], factory, faiss.METRIC_INNER_PRODUCT)
    if not index.is_trained:
        index.train(A)
    index.add(A)
    build_s = time.perf_counter() - t0
    if nprobe:
        index.nprobe = nprobe
    lat = []
    for s in range(0, 512, 32):
        t0 = time.perf_counter()
        index.search(Uq[s:s + 32], 100)
        lat.append((time.perf_counter() - t0) / 32 * 1000)
    _, I = index.search(Uq, 100)
    hits = [np.isin(t, I[i]).sum() / len(t) for i, t in enumerate(q_truth)]
    record("1_index", index=name, build_s=round(build_s, 2),
           index_mb=round(faiss.serialize_index(index).nbytes / 1e6, 1),
           ms_per_query_p50=round(float(np.median(lat)), 3),
           ms_per_query_p99=round(float(np.percentile(lat, 99)), 3),
           recall_at_100=round(float(np.mean(hits)), 5))
    del index
    gc.collect()


faiss_bench("FAISS IndexFlatIP", "Flat")
faiss_bench("FAISS IVF256 nprobe=8", "IVF256,Flat", nprobe=8)
faiss_bench("FAISS IVF256 nprobe=32", "IVF256,Flat", nprobe=32)
faiss_bench("FAISS HNSW32", "HNSW32")


print("=" * 70)
print("EXPERIMENT 2 — per-pair similarity: exact sparse vs 64-d dense")
print("=" * 70)
rng = np.random.default_rng(0)
NP = 1_000_000
qi = rng.integers(0, Q.shape[0], NP)
ai = rng.integers(0, N_ART, NP)

def sparse_pairs():
    out = np.empty(NP, dtype=np.float32)
    for s in range(0, NP, 250_000):
        e = min(s + 250_000, NP)
        out[s:e] = np.asarray(Q[qi[s:e]].multiply(BM_W[ai[s:e]]).sum(axis=1)).ravel()
    return out

def dense_pairs():
    out = np.empty(NP, dtype=np.float32)
    for s in range(0, NP, 250_000):
        e = min(s + 250_000, NP)
        out[s:e] = np.einsum("ij,ij->i", Uq[qi[s:e] % len(Uq)], A[ai[s:e]], optimize=True)
    return out

m0 = rss_mb(); t_sp, _ = timeit(sparse_pairs, repeat=2); mem_sp = rss_mb() - m0
gc.collect()
m0 = rss_mb(); t_de, _ = timeit(dense_pairs, repeat=2); mem_de = rss_mb() - m0
record("2_pairwise", method="exact sparse BM25", s_per_1M_pairs=round(t_sp, 2),
       delta_rss_mb=round(mem_sp, 1), pairs_per_sec=int(NP / t_sp))
record("2_pairwise", method="dense 64-d einsum", s_per_1M_pairs=round(t_de, 2),
       delta_rss_mb=round(mem_de, 1), pairs_per_sec=int(NP / t_de))
record("2_pairwise", method="RATIO sparse/dense", s_per_1M_pairs=round(t_sp / t_de, 1),
       delta_rss_mb="-", pairs_per_sec="-")


print("=" * 70)
print("EXPERIMENT 3 — scoring throughput vs pair budget (memory/latency curve)")
print("=" * 70)
for budget in [375_000, 750_000, 1_500_000, 3_000_000]:
    rows = max(1, int(budget / 40))
    n_pairs = n_imp = 0
    gc.collect()
    peak = rss_mb()
    t0 = time.perf_counter()
    for ch in iter_slices(beh_eval, rows):
        imp, pos, code, X, _ = featurise(ch, ctx_eval)
        s = model.predict(X[:, IDX_FULL], num_iteration=model.best_iteration)
        n_pairs += len(s); n_imp += ch.height
        peak = max(peak, rss_mb())
        del imp, pos, code, X, s
        if n_imp >= 40_000:
            break
    el = time.perf_counter() - t0
    record("3_throughput", pair_budget=budget, rows_per_chunk=rows,
           pairs_per_sec=int(n_pairs / el), impressions_per_sec=int(n_imp / el),
           peak_rss_mb=round(peak, 0),
           est_test_minutes=round(150_000_000 / (n_pairs / el) / 60, 1))
    gc.collect()


print("=" * 70)
print("EXPERIMENT 4 — embedding dimensionality: cost vs accuracy")
print("=" * 70)
# PCA components are ordered, so EMB[:, :k] is the top-k subspace (re-normalised).
DIMS = sorted({k for k in [8, 16, 32, 64, EMB.shape[1]] if k <= EMB.shape[1]})
for k in DIMS:
    Ak = EMB[:, :k].copy()
    Ak /= np.maximum(np.linalg.norm(Ak, axis=1, keepdims=True), 1e-8)
    Uk = Uq[:, :k].copy()
    Uk /= np.maximum(np.linalg.norm(Uk, axis=1, keepdims=True), 1e-8)
    t_pair, _ = timeit(lambda: np.einsum("ij,ij->i", Uk[qi[:500_000] % len(Uk)],
                                         Ak[ai[:500_000]], optimize=True), repeat=2)
    rec = recall_at_k_dense(Uk, Ak, q_truth, None, ks=(100,))[100]
    record("4_dimension", dim=k, catalogue_mb=round(Ak.nbytes / 1e6, 1),
           s_per_500k_pairs=round(t_pair, 3), recall_at_100=round(rec, 5))


print("=" * 70)
print("EXPERIMENT 5 — Polars vs pandas on the hot path (explode + group-by)")
print("=" * 70)
sample = beh_eval.head(200_000).select("impression_id", "article_ids_inview")

def polars_path():
    return (sample.select("article_ids_inview").explode("article_ids_inview")
            .group_by("article_ids_inview").agg(pl.len()).height)

t_pl, _ = timeit(polars_path, repeat=3)
try:
    import pandas as pd
    pdf = sample.to_pandas()

    def pandas_path():
        return pdf["article_ids_inview"].explode().value_counts().shape[0]

    t_pd, _ = timeit(pandas_path, repeat=3)
except Exception as e:  # noqa: BLE001
    t_pd = float("nan")
    print("  pandas path failed:", e)

record("5_dataframe", library="polars", s_per_200k_rows=round(t_pl, 3))
record("5_dataframe", library="pandas", s_per_200k_rows=round(t_pd, 3))
record("5_dataframe", library="RATIO pandas/polars", s_per_200k_rows=round(t_pd / t_pl, 1))


bench = pl.DataFrame(BENCH, infer_schema_length=None)
bench.write_csv(Path(CFG["out_dir"]) / "engineering_benchmarks.csv")

# print one clean sub-table per experiment (drop the all-null columns)
for exp in bench["experiment"].unique(maintain_order=True):
    sub = bench.filter(pl.col("experiment") == exp).drop("experiment")
    sub = sub[[c for c in sub.columns if sub[c].null_count() < sub.height]]
    print(f"\n----- {exp} -----")
    with pl.Config(tbl_cols=-1, tbl_width_chars=200):
        print(sub)
print("\nwrote engineering_benchmarks.csv")

EXPERIMENT 1 — retrieval index: build cost vs query latency vs recall
query set: 3,000 impressions | catalogue 125,541 × 64d
  1_index: index=BM25 sparse CSR  build_s=4.22  index_mb=20.8  ms_per_query_p50=5.755  ms_per_query_p99=6.183  recall_at_100=0.00707
  1_index: index=numpy brute force  build_s=0.0  index_mb=32.1  ms_per_query_p50=0.807  ms_per_query_p99=0.875  recall_at_100=0.00667
  1_index: index=FAISS IndexFlatIP  build_s=0.01  index_mb=32.1  ms_per_query_p50=1.004  ms_per_query_p99=1.096  recall_at_100=0.00667
  1_index: index=FAISS IVF256 nprobe=8  build_s=0.3  index_mb=33.2  ms_per_query_p50=0.052  ms_per_query_p99=0.058  recall_at_100=0.00733
  1_index: index=FAISS IVF256 nprobe=32  build_s=0.29  index_mb=33.2  ms_per_query_p50=0.138  ms_per_query_p99=0.165  recall_at_100=0.00667
  1_index: index=FAISS HNSW32  build_s=3.61  index_mb=66.3  ms_per_query_p50=0.029  ms_per_query_p99=0.039  recall_at_100=0.006
EXPERIMENT 2 — per-pair similarity: exact sparse vs 64-d dense
  2_